# v6 overlap40 · Gated-ReZero A′ · seed42
零初始化门控残差，全量训练，physical batch=4，仅使用 Train/Val，不评估 Test。

In [ ]:
from pathlib import Path
import json, os, sys
REPO_DIR = Path('/root/autodl-tmp/projects/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
DATA_ROOT = Path('/root/autodl-tmp/datasets/dataset_v6_random811_overlap40')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
HF_CACHE_SOURCE = Path('/root/autodl-tmp/resnet50_imagenet_cache')
CONFIG_PATH = PROJECT_DIR / 'configs' / 'v6_overlap40_gated_rezero_batch4_seed42.json'
assert REPO_DIR.is_dir() and CONFIG_PATH.is_file()
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['module'] == 'gated_rezero' and config['seed'] == 42
print('配置:', CONFIG_PATH)
print('输出:', OUTPUT_ROOT / f"result_{config['run_name']}")

In [ ]:
import importlib
required = ['torch', 'segmentation_models_pytorch', 'rasterio', 'matplotlib', 'tqdm']
missing = []
for name in required:
    try:
        importlib.import_module(name)
    except Exception as exc:
        missing.append((name, repr(exc)))
assert not missing, f'缺少运行依赖: {missing}'
print('依赖检查通过')

In [ ]:
import subprocess
command = [sys.executable, str(PROJECT_DIR / 'scripts' / 'run_autodl_gated_rezero.py'),
           '--project-dir', str(PROJECT_DIR), '--config', str(CONFIG_PATH),
           '--data-dir', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT),
           '--hf-cache-source', str(HF_CACHE_SOURCE)]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
assert process.wait() == 0, '训练进程失败'

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
result = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert result['test_evaluated'] is False
print(json.dumps(result, ensure_ascii=False, indent=2))
print('下载压缩包:', Path(str(result_dir) + '.zip'))